In [1]:
# Cell 2 — Imports

import os
import json
from pathlib import Path
from typing import Optional, List

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

from pymongo import MongoClient
import pymupdf
import pandas as pd

print("Imports completed.")


C:\Users\Abhist\AppData\Local\Temp\ipykernel_18932\637097953.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports completed.


In [2]:
# Cell 3 — Load API key

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found. "
        "Make sure your .env file contains GROQ_API_KEY=your_key"
    )

print("Groq API key loaded successfully.")


Groq API key loaded successfully.


In [4]:
# Cell 4 — Configure the 51-PDF input folder

# IMPORTANT:
# Use Path(), not a normal string, because the notebook uses
# .exists(), .is_dir(), .glob(), and .resolve().

PDF_FOLDER = Path(r"C:\Users\Abhist\Desktop\KOHLER\kohler_pdfs")

if not PDF_FOLDER.exists():
    raise FileNotFoundError(
        f"PDF folder not found: {PDF_FOLDER.resolve()}\n"
        "Check that the folder path is correct."
    )

if not PDF_FOLDER.is_dir():
    raise NotADirectoryError(
        f"The specified path is not a folder: {PDF_FOLDER.resolve()}"
    )

# Find both .pdf and .PDF files
pdf_files = sorted(
    list(PDF_FOLDER.glob("*.pdf")) +
    list(PDF_FOLDER.glob("*.PDF"))
)

# Remove duplicates if a filesystem is case-insensitive
pdf_files = list(dict.fromkeys(pdf_files))

print(f"PDF folder: {PDF_FOLDER.resolve()}")
print(f"PDF files found: {len(pdf_files)}")

if len(pdf_files) != 51:
    print(f"WARNING: Expected 51 PDFs, but found {len(pdf_files)}.")
else:
    print("✓ All 51 KOHLER PDFs found successfully.")


PDF folder: C:\Users\Abhist\Desktop\KOHLER\kohler_pdfs
PDF files found: 51
✓ All 51 KOHLER PDFs found successfully.


In [5]:
# Cell 5 — Product data schema

class Dimensions(BaseModel):
    width_mm: Optional[float] = None
    depth_mm: Optional[float] = None
    height_mm: Optional[float] = None


class Installation(BaseModel):
    type: Optional[str] = None
    waste_outlet: Optional[str] = None
    rough_in_mm: Optional[float] = None


class Electrical(BaseModel):
    required: Optional[bool] = None
    voltage: Optional[str] = None
    power_w: Optional[float] = None


class Product(BaseModel):
    product_id: Optional[str] = None
    product_name: Optional[str] = None
    category: Optional[str] = None
    subcategory: Optional[str] = None
    collection: Optional[str] = None

    dimensions: Dimensions = Field(default_factory=Dimensions)
    installation: Installation = Field(default_factory=Installation)
    electrical: Electrical = Field(default_factory=Electrical)

    features: List[str] = Field(default_factory=list)
    color: List[str] = Field(default_factory=list)
    material: Optional[str] = None


print("Product schema created.")


Product schema created.


In [6]:
# Cell 6 — Initialize Groq LLM

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

print("Groq LLM initialized.")


Groq LLM initialized.


In [7]:
# Cell 7 — Extraction prompt

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        '''
You are a product data extraction system for a KOHLER bathroom product catalog.

Your task is to extract structured information from a KOHLER product
specification document.

IMPORTANT RULES:

1. Extract ONLY information explicitly present in the provided document.
2. NEVER invent or guess information.
3. If information is not available, return null.
4. Preserve the product/model number exactly as written.
5. Convert dimensions to millimetres when the conversion is unambiguous.
6. Convert power to watts when the conversion is unambiguous.
7. Extract product features explicitly mentioned in the document.
8. Keep installation information separate from product features.
9. Extract electrical requirements only when explicitly stated.
10. Extract the product colour when explicitly stated, including values such as
    White, Polished Chrome, Matte Black, etc.
11. Do NOT infer product price.
12. Do NOT infer aesthetic or theme scores.
13. Do NOT decide whether the product fits in a bathroom.
14. Do NOT make recommendations.
15. Do NOT use outside knowledge.
16. If a value cannot be confidently extracted from the document, return null.
17. Pay special attention to technical specifications and dimension information
    written as text.
18. Do not confuse width, depth, and height.
19. If the document gives dimensions but their orientation is not clear enough
    to assign them safely to width/depth/height, do not guess.

Return the information according to the Product schema.
'''
    ),
    (
        "human",
        '''
Extract the product information from the following KOHLER specification document:

---------------- DOCUMENT ----------------

{document}

---------------- END DOCUMENT ----------------
'''
    )
])

print("Extraction prompt created.")


Extraction prompt created.


In [8]:
# Cell 8 — Create structured-output Groq chain

structured_llm = llm.with_structured_output(Product)

chain = prompt | structured_llm

print("Structured extraction chain created.")


Structured extraction chain created.


In [9]:
# Cell 9 — Extract text from a PDF

def extract_pdf_text(pdf_path):
    '''
    Extract selectable text from all pages of a PDF.
    '''
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()

    text = "\n".join(
        doc.page_content
        for doc in pages
    )

    return text


print("PDF text extraction function ready.")


PDF text extraction function ready.


In [10]:
# Cell 10 — Test PDF text extraction

if len(pdf_files) == 0:
    raise ValueError("No PDF files found.")

test_pdf = pdf_files[0]

test_text = extract_pdf_text(test_pdf)

print("Test PDF:", test_pdf.name)
print("Extracted characters:", len(test_text))
print()
print(test_text[:3000])


Test PDF: 1408991-IN4.pdf
Extracted characters: 2676

1408991-IN4-A
 
Features
• Ergonomic and Straight line design suitable for compact 
 rooms with deck space. 
• Easy installation as it requires counter cutting to 
 accommodate drain only.
• Optimum Depth to contain Splashes.
•  Above counter without faucet deck.
• Only drain cutting(no profile cutting required).
Product Details
• Material: Vitreous China
• Overflow: Without Overflow Hole
Recommended Accessories 
•  Bottle Trap (75823IN-**).
•  Compatible Drain (20746IN-**).
Code DescriptionColour
0 White
Available Colour/Finishes
Colour tiles intended for reference only.
Customer Care: 1800-103-2244 (Toll Free) & +91-124-4319685 / 86
Kohler Co. reserves the right to make revisions without notice to product speciﬁcations.
For the most current Speciﬁcation Sheet, go to www.kohler.co.in
SPAN®
Square Vessel Without Deck ( Small)
K-31459IN
KOHLER® Ten-Year Limited Warranty
See website for detailed warranty information
www.kohler.co.in/w

In [11]:
# Cell 11 — Extract one product using Groq

def extract_product_from_text(pdf_path):
    '''
    Extract one structured Product object from one PDF.
    '''
    text = extract_pdf_text(pdf_path)

    if not text.strip():
        raise ValueError(
            f"No selectable text found in PDF: {pdf_path.name}"
        )

    product = chain.invoke({
        "document": text
    })

    return product


test_product = extract_product_from_text(test_pdf)

print(test_product.model_dump_json(indent=2))


{
  "product_id": "1408991-IN4-A",
  "product_name": "SPAN® Square Vessel Without Deck ( Small)",
  "category": null,
  "subcategory": null,
  "collection": "SPAN®",
  "dimensions": {
    "width_mm": 344.0,
    "depth_mm": 483.0,
    "height_mm": 141.0
  },
  "installation": {
    "type": "Counter cutting",
    "waste_outlet": null,
    "rough_in_mm": null
  },
  "electrical": {
    "required": null,
    "voltage": null,
    "power_w": null
  },
  "features": [
    "Ergonomic and Straight line design suitable for compact rooms with deck space.",
    "Easy installation as it requires counter cutting to accommodate drain only.",
    "Optimum Depth to contain Splashes.",
    "Above counter without faucet deck.",
    "Only drain cutting (no profile cutting required).",
    "Without Overflow Hole"
  ],
  "color": [
    "White"
  ],
  "material": "Vitreous China"
}


In [12]:
# Cell 12 — Validate one extracted product

def validate_product(product):
    '''
    Check important fields without modifying the extracted product.
    '''
    problems = []

    if not product.product_id:
        problems.append("Missing product ID")

    if not product.product_name:
        problems.append("Missing product name")

    if not product.category:
        problems.append("Missing category")

    if product.dimensions.width_mm is None:
        problems.append("Missing width")

    if product.dimensions.depth_mm is None:
        problems.append("Missing depth")

    if product.dimensions.height_mm is None:
        problems.append("Missing height")

    return problems


test_problems = validate_product(test_product)

if test_problems:
    print("Validation warnings:")
    for problem in test_problems:
        print("-", problem)
else:
    print("✓ Product passed validation.")


Validation warnings:
- Missing category


In [13]:
# Cell 13 — MongoDB connection

# Start your local MongoDB server before running this cell.
# MongoDB Compass connects to the same server using:
# mongodb://localhost:27017

MONGO_URI = "mongodb://localhost:27017"

try:
    mongo_client = MongoClient(
        MONGO_URI,
        serverSelectionTimeoutMS=5000
    )

    # Test the connection
    mongo_client.admin.command("ping")

    db = mongo_client["kohler_ai_bathroom"]
    products_collection = db["products"]

    # Product IDs should be unique
    products_collection.create_index(
        "product_id",
        unique=True
    )

    print("✓ MongoDB connected successfully.")
    print("Database: kohler_ai_bathroom")
    print("Collection: products")

except Exception as e:
    raise ConnectionError(
        "Could not connect to MongoDB at mongodb://localhost:27017. "
        "Make sure your local MongoDB server is running."
    ) from e


✓ MongoDB connected successfully.
Database: kohler_ai_bathroom
Collection: products


In [14]:
# Cell 14 — Convert Product object to MongoDB document

def product_to_document(product, pdf_path, extraction_status="success", validation_problems=None):
    '''
    Convert the Pydantic Product object into a MongoDB-ready dictionary.
    '''
    document = product.model_dump()

    document["source_pdf"] = pdf_path.name
    document["source_type"] = "KOHLER specification PDF"
    document["extraction_method"] = "Groq text extraction"
    document["extraction_status"] = extraction_status
    document["validation_problems"] = validation_problems or []

    return document


test_document = product_to_document(
    test_product,
    test_pdf,
    validation_problems=test_problems
)

print(json.dumps(test_document, indent=2, default=str))


{
  "product_id": "1408991-IN4-A",
  "product_name": "SPAN\u00ae Square Vessel Without Deck ( Small)",
  "category": null,
  "subcategory": null,
  "collection": "SPAN\u00ae",
  "dimensions": {
    "width_mm": 344.0,
    "depth_mm": 483.0,
    "height_mm": 141.0
  },
  "installation": {
    "type": "Counter cutting",
    "waste_outlet": null,
    "rough_in_mm": null
  },
  "electrical": {
    "required": null,
    "voltage": null,
    "power_w": null
  },
  "features": [
    "Ergonomic and Straight line design suitable for compact rooms with deck space.",
    "Easy installation as it requires counter cutting to accommodate drain only.",
    "Optimum Depth to contain Splashes.",
    "Above counter without faucet deck.",
    "Only drain cutting (no profile cutting required).",
    "Without Overflow Hole"
  ],
  "color": [
    "White"
  ],
  "material": "Vitreous China",
  "source_pdf": "1408991-IN4.pdf",
  "source_type": "KOHLER specification PDF",
  "extraction_method": "Groq text extra

In [15]:
# Cell 15 — Upsert one product into MongoDB

def upsert_product(document):
    '''
    Insert a product or update it if the same product_id already exists.

    This prevents duplicate documents when the notebook is rerun.
    '''
    product_id = document.get("product_id")

    if not product_id:
        raise ValueError(
            "Cannot upsert product because product_id is missing."
        )

    result = products_collection.update_one(
        {"product_id": product_id},
        {"$set": document},
        upsert=True
    )

    return result


result = upsert_product(test_document)

print("Test product saved to MongoDB.")
print("Matched:", result.matched_count)
print("Modified:", result.modified_count)
print("Upserted ID:", result.upserted_id)


Test product saved to MongoDB.
Matched: 0
Modified: 0
Upserted ID: 6aa95896c877dad10aa25e69


## Process all 51 PDFs

The following cell processes every PDF individually.

For each PDF:

1. Extract PDF text.
2. Send the text to Groq.
3. Convert the response into the `Product` schema.
4. Validate the extracted information.
5. Add source metadata.
6. Upsert the product into MongoDB.
7. Record success or failure.

A failure on one PDF will **not stop the remaining PDFs**.


In [16]:
# Cell 16 — Process all PDFs

all_results = []
failed_pdfs = []

print(f"Starting extraction for {len(pdf_files)} PDFs...")
print("=" * 70)

for index, pdf_path in enumerate(pdf_files, start=1):

    print(f"[{index}/{len(pdf_files)}] Processing: {pdf_path.name}")

    try:
        product = extract_product_from_text(pdf_path)

        validation_problems = validate_product(product)

        document = product_to_document(
            product,
            pdf_path,
            extraction_status="success",
            validation_problems=validation_problems
        )

        upsert_product(document)

        all_results.append({
            "pdf": pdf_path.name,
            "product_id": product.product_id,
            "product_name": product.product_name,
            "category": product.category,
            "status": "success",
            "validation_problems": "; ".join(validation_problems)
        })

        if validation_problems:
            print("  ✓ Extracted with validation warnings.")
        else:
            print("  ✓ Extracted successfully.")

    except Exception as e:

        error_message = str(e)

        print(f"  ✗ FAILED: {error_message}")

        failed_pdfs.append({
            "pdf": pdf_path.name,
            "error": error_message
        })

        all_results.append({
            "pdf": pdf_path.name,
            "product_id": None,
            "product_name": None,
            "category": None,
            "status": "failed",
            "validation_problems": error_message
        })

print("=" * 70)
print("Extraction completed.")
print(f"Successful PDFs: {len(pdf_files) - len(failed_pdfs)}")
print(f"Failed PDFs: {len(failed_pdfs)}")


Starting extraction for 51 PDFs...
[1/51] Processing: 1408991-IN4.pdf
  ✓ Extracted with validation warnings.
[2/51] Processing: 29024IN-1_Chalice w deck vessel.pdf
  ✓ Extracted successfully.
[3/51] Processing: 90011T.pdf
  ✓ Extracted with validation warnings.
[4/51] Processing: K-12925IN_spec_IN_Kohler_en.pdf
  ✗ FAILED: Error code: 400 - {'error': {'message': 'Tool call validation failed: tool call validation failed: parameters for tool Product did not match schema: errors: [`/dimensions`: expected object, but got null, `/electrical`: expected object, but got null]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "Product", "arguments": {\n  "category": "Faucet",\n  "collection": "Complementary™",\n  "product_id": "K-12925IN",\n  "product_name": "Complementary™ Health faucet, 11.7 lpm",\n  "subcategory": "Health faucet",\n  "color": ["Polished Chrome", "Vibrant® French Gold", "Vibrant Rose Gold"],\n  "dimensions": null,\n  "electrical": n

In [17]:
# Cell 17 — Create extraction report

extraction_report = pd.DataFrame(all_results)

print("Extraction report:")
display(extraction_report)

if failed_pdfs:
    print("\nFailed PDFs:")
    display(pd.DataFrame(failed_pdfs))
else:
    print("\n✓ No PDFs failed.")


Extraction report:


,pdf,product_id,product_name,category,status,validation_problems
0,1408991-IN4.pdf,1408991-IN4-A,SPAN® Square Vessel Without Deck ( Small),NaN,success,Missing category
1,29024IN-1_Chalice w deck vessel.pdf,29024IN-1,CHALICE ROUND VESSEL 1TAP HOLE,Vessel,success,
2,90011T.pdf,1311520-A04-D,Mica® Square 393mm Basin,BASINS / COUNTERTOP,success,Missing height
3,K-12925IN_spec_IN_Kohler_en.pdf,NaN,NaN,NaN,failed,Error code: 400 - {'error': {'message': 'Tool ...
4,K-12927IN_spec_IN_Kohler_en.pdf,K-12927IN,Complimentary™ Hygiene Spray,NaN,success,Missing category; Missing width; Missing depth...
5,K-1381T-S_spec_CN_Kohler_en.pdf,NaN,NaN,NaN,failed,Error code: 400 - {'error': {'message': 'Tool ...
6,K-1381T-S_spec_IN_Kohler_en.pdf,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,success,Missing width; Missing depth
7,K-15848T_spec_IN_Kohler_en.pdf,NaN,NaN,NaN,failed,Error code: 400 - {'error': {'message': 'Tool ...
8,K-17629T-NS_spec_CN_Kohler_en (1).pdf,NaN,NaN,NaN,failed,Error code: 400 - {'error': {'message': 'Tool ...
9,K-17629T-NS_spec_CN_Kohler_en.pdf,NaN,NaN,NaN,failed,Error code: 400 - {'error': {'message': 'Tool ...



Failed PDFs:


,pdf,error
0,K-12925IN_spec_IN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...
1,K-1381T-S_spec_CN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...
2,K-15848T_spec_IN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...
3,K-17629T-NS_spec_CN_Kohler_en (1).pdf,Error code: 400 - {'error': {'message': 'Tool ...
4,K-17629T-NS_spec_CN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...
5,K-1855IN_spec_IN_Kohler_en.pdf,Error code: 429 - {'error': {'message': 'Rate ...
6,K-2215IN-2_spec_IN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...
7,K-23975IN-4ND_spec_IN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...
8,K-25318IN_spec_IN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...
9,K-27484IN-4_spec_IN_Kohler_en.pdf,Error code: 400 - {'error': {'message': 'Tool ...


In [18]:
# Cell 18 — Save extraction report as CSV

report_path = Path("kohler_extraction_report.csv")

extraction_report.to_csv(
    report_path,
    index=False
)

print(f"Report saved to: {report_path.resolve()}")


Report saved to: C:\Users\Abhist\Desktop\KOHLER\scripts\kohler_extraction_report.csv


In [19]:
# Cell 19 — Verify MongoDB contents

total_documents = products_collection.count_documents({})

print(f"Products currently stored in MongoDB: {total_documents}")

print("\nSample documents:")

sample_documents = list(
    products_collection.find(
        {},
        {
            "_id": 0,
            "product_id": 1,
            "product_name": 1,
            "category": 1,
            "dimensions": 1,
            "source_pdf": 1
        }
    ).limit(5)
)

for document in sample_documents:
    print(json.dumps(document, indent=2, default=str))


Products currently stored in MongoDB: 36

Sample documents:
{
  "product_id": "1408991-IN4-A",
  "category": null,
  "dimensions": {
    "width_mm": 344.0,
    "depth_mm": 483.0,
    "height_mm": 141.0
  },
  "product_name": "SPAN\u00ae Square Vessel Without Deck ( Small)",
  "source_pdf": "1408991-IN4.pdf"
}
{
  "product_id": "29024IN-1",
  "category": "Vessel",
  "dimensions": {
    "width_mm": 470.0,
    "depth_mm": 116.0,
    "height_mm": 139.0
  },
  "product_name": "CHALICE ROUND VESSEL 1TAP HOLE",
  "source_pdf": "29024IN-1_Chalice w deck vessel.pdf"
}
{
  "product_id": "1311520-A04-D",
  "category": "BASINS / COUNTERTOP",
  "dimensions": {
    "width_mm": 393.0,
    "depth_mm": 393.0,
    "height_mm": null
  },
  "product_name": "Mica\u00ae Square 393mm Basin",
  "source_pdf": "90011T.pdf"
}
{
  "product_id": "K-12927IN",
  "category": null,
  "dimensions": {
    "width_mm": null,
    "depth_mm": null,
    "height_mm": null
  },
  "product_name": "Complimentary\u2122 Hygiene Sp

In [20]:
# Cell 20 — Category summary

category_summary = list(
    products_collection.aggregate([
        {
            "$group": {
                "_id": "$category",
                "count": {"$sum": 1}
            }
        },
        {
            "$sort": {
                "count": -1
            }
        }
    ])
)

print("Products by category:")

for item in category_summary:
    print(
        f"{item['_id']}: {item['count']}"
    )


Products by category:
Faucet: 14
Bathroom sink: 5
Toilet: 5
None: 4
Sink: 2
Toilet Seat: 1
Vessel: 1
BASINS / COUNTERTOP: 1
Bathroom Sink: 1
Faucet Trim: 1
Bathroom sink faucet: 1


In [21]:
# Cell 21 — Find products with missing dimensions

missing_dimension_documents = list(
    products_collection.find(
        {
            "$or": [
                {"dimensions.width_mm": None},
                {"dimensions.depth_mm": None},
                {"dimensions.height_mm": None}
            ]
        },
        {
            "_id": 0,
            "product_id": 1,
            "product_name": 1,
            "category": 1,
            "dimensions": 1,
            "source_pdf": 1
        }
    )
)

print(
    f"Products with one or more missing dimensions: "
    f"{len(missing_dimension_documents)}"
)

if missing_dimension_documents:
    display(
        pd.DataFrame(missing_dimension_documents)
    )


Products with one or more missing dimensions: 34


,product_id,category,dimensions,product_name,source_pdf
0,1311520-A04-D,BASINS / COUNTERTOP,"{'width_mm': 393.0, 'depth_mm': 393.0, 'height...",Mica® Square 393mm Basin,90011T.pdf
1,K-12927IN,NaN,"{'width_mm': None, 'depth_mm': None, 'height_m...",Complimentary™ Hygiene Spray,K-12927IN_spec_IN_Kohler_en.pdf
2,K-1381T-S,Toilet,"{'width_mm': None, 'depth_mm': None, 'height_m...",Veil™ One-piece elongated toilet with skirted ...,K-1381T-S_spec_IN_Kohler_en.pdf
3,K-17629T-NS,Toilet,"{'width_mm': None, 'depth_mm': None, 'height_m...",Ove™ One-piece round-front toilet with skirted...,K-17629T-NS_spec_IN_Kohler_en.pdf
4,K-17660T-M,Toilet Seat,"{'width_mm': None, 'depth_mm': None, 'height_m...",Ove™ Quiet-Close™ elongated toilet seat,K-17660T-M_spec_IN_Kohler_en.pdf
5,K-1851IN,NaN,"{'width_mm': None, 'depth_mm': None, 'height_m...",Brive Plus,K-1851IN_spec_IN_Kohler_en.pdf
6,K-1853IN,NaN,"{'width_mm': None, 'depth_mm': None, 'height_m...",Brive Plus,K-1853IN_spec_IN_Kohler_en.pdf
7,K-21226IN,Sink,"{'width_mm': 600.0, 'depth_mm': 113.0, 'height...",ModernLife Edge 600 mm rectangular vessel bath...,K-21226IN_spec_IN_Kohler_en.pdf
8,K-2200IN,Sink,"{'width_mm': 413.0, 'depth_mm': None, 'height_...",Conical Bell™,K-2200IN_spec_IN_Kohler_en.pdf
9,K-23486IN-4ND,Faucet,"{'width_mm': None, 'depth_mm': None, 'height_m...","Parallel™ Wall-mount bathroom sink faucet, 9.0...",K-23486IN-4ND_spec_IN_Kohler_en.pdf
